# Compensation distributions
**Jamal Ford — Part B — notebook 02**

**Dataset status:** Uses Samia's cleaned GitHub dataset with **4,435 respondents**. The source is `data/processed/us_fulltime_clean.csv`, prepared in notebook 01 at GitHub commit `02b883a`. This is the current team-data analysis.

This notebook describes annual total compensation, professional work experience,
roles, work arrangements, and education using the processed CSV from notebook 01.
One row is one respondent. It does not re-clean raw data, impute experience, change
role families, or remove any additional compensation observations.

The 2025 employment question does not directly identify full-time work. Notebook 01
keeps U.S. employed respondents who report compensation and meet its pay cutoffs;
it does not filter on professional-developer identity. We describe that prepared
sample accurately rather than infer full-time status from the filename.

The input includes natural-log pay (`log_pay`), nine role families, and three work
arrangements. Hybrid includes the original Your choice response, as defined by
notebook 01. The original five work options are also shown below. The prepared
band named `20+` contains values above 20; charts label the same members **21+**.
See [notebook 01](01_data_prep.ipynb) for population and preparation decisions.

**Shared helper:** [eda_common.py](../scripts/eda_common.py) contains repeated input checks, summaries, and chart formatting. Keep the scripts and data folders with these notebooks. In VS Code, open the whole project folder, restart the kernel, and run all cells.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

# Locate the team files from the notebook folder, project folder, or workspace root.
ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for folder in [candidate, candidate / "Project/Developer_Compensation",
                   candidate / "Developer_Compensation", candidate / "Project"]:
        if (folder / "scripts/eda_common.py").exists() and (folder / "data/docs/team_source.json").exists():
            ROOT = folder
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project folder with its data and scripts.")
sys.path.insert(0, str(ROOT / "scripts"))
import importlib
import eda_common
importlib.reload(eda_common)
from eda_common import (
    load_data, pay_summary, band_order, count_plot, box_plot, dollars,
    wrapped, save_figure, MIN_GROUP_N, MISSING,
)
df, info, output = load_data(ROOT)
pd.set_option("display.max_columns", 30)
display(Markdown(f"**Prepared team sample: {len(df):,} U.S. employed respondents.** "
                 "All input rows are retained in this EDA."))
display(pd.Series(info, name="Run information").to_frame())


## Sample coverage and missing values
Missing categories remain visible in counts and pay tables. Numerical experience
plots use respondents with reported experience. In 2025 a blank experience response
can mean zero years or nonresponse, so it is not replaced with zero.


In [ ]:
missing = pd.DataFrame([
    {"variable": "Professional work experience", "missing_n": df.ExperienceYears.isna().sum()},
    {"variable": "Work arrangement", "missing_n": df.WorkArrangement.eq(MISSING).sum()},
    {"variable": "Education", "missing_n": df.Education.eq(MISSING).sum()},
    {"variable": "Role family", "missing_n": df.RoleGroup.eq(MISSING).sum()},
])
missing["missing_pct"] = 100*missing["missing_n"]/len(df)
missing.to_csv(output["tables"] / "eda_missing_values.csv", index=False)
display(missing.round(2))


## Compensation summary
Pay is annual total compensation in USD, including bonuses and perks before taxes.
Q1 and Q3 bound the middle half of respondents; IQR is Q3 minus Q1. The median is
less affected by the upper tail than the mean. Summary statistics use the prepared
sample after notebook 01's lower and upper pay cutoffs.


In [ ]:
overall = pay_summary(df)
overall["mean_usd"] = df.PayUSD.mean()
overall["min_usd"] = df.PayUSD.min()
overall["p99_usd"] = df.PayUSD.quantile(.99)
overall["max_usd"] = df.PayUSD.max()
overall.to_csv(output["tables"] / "overall_compensation.csv", index=False)
display(overall.round(2))
row = overall.iloc[0]
display(Markdown(
    f"Median total compensation is **{dollars(row.median_usd)}**; the middle half spans "
    f"**{dollars(row.q1_usd)} to {dollars(row.q3_usd)}** (IQR **{dollars(row.iqr_usd)}**). "
    f"The mean is **{dollars(row.mean_usd)}**. "
    "A higher mean than median is consistent with the right tail visible in the histogram."
))


## Compensation on raw and natural log scales
Both panels include all retained respondents. The natural-log panel gives more
space to differences among common pay values. A one-unit increase in ln pay is
a multiplication by e, about 2.72. We use the natural-log field supplied by notebook 01,
which is also the modeling target; this is not a log10 axis.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].hist(df.PayUSD, bins=40, color="#31688e", edgecolor="white")
axes[0].xaxis.set_major_formatter(FuncFormatter(dollars))
axes[0].tick_params(axis="x", labelrotation=25)
axes[0].set(title=f"Annual total compensation (n={len(df):,})",
            xlabel="Annual total compensation in USD", ylabel="Respondents")
axes[1].hist(df.LogPay, bins=35, color="#35a28c", edgecolor="white")
axes[1].set(title="Same respondents, natural-log compensation",
            xlabel="ln annual total compensation in USD", ylabel="Respondents")
save_figure(fig, "pay_distribution", info, output)


## Professional work experience
`WorkExp_num` measures professional work experience generally, not years coding
professionally. The histogram uses one-year bins because responses are rounded
to whole years. The band memberships are inherited from notebook 01 without changes.


In [ ]:
experience = df.ExperienceYears.dropna()
experience.describe().to_frame("professional_work_years").to_csv(output["tables"] / "experience_summary.csv")
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.hist(experience, bins=np.arange(-.5, np.ceil(experience.max())+1.5, 1),
        color="#31688e", edgecolor="white")
ax.set(title=f"Professional work experience (reported n={len(experience):,}; missing n={df.ExperienceYears.isna().sum():,})",
       xlabel="Years of professional work experience", ylabel="Respondents")
save_figure(fig, "experience_distribution", info, output)
display(Markdown(f"Reported experience has a median of **{experience.median():g} years**. "
                 f"**{df.ExperienceYears.isna().sum():,}** respondents have missing experience."))


## Roles work arrangements education and experience bands
The bar charts count respondents, with an explicit missing category where needed.
Role families and three-way work arrangements are taken directly from notebook 01.
The additional detailed-work chart preserves the original five choices for context.
Education wording is shortened for display; education categories are not combined.


In [ ]:
for column, title, filename in [
    ("RoleGroup", "Respondents by role family", "counts_role"),
    ("WorkArrangement", "Respondents by prepared work arrangement", "counts_RemoteWork"),
    ("Education", "Respondents by education", "counts_EdLevel"),
    ("ExperienceBand", "Respondents by experience band", "counts_exp_band"),
    ("WorkArrangementDetail", "Respondents by original work arrangement", "counts_RemoteWork_detail"),
]:
    count_plot(df, column, title, filename, info, output)
role_counts = df.RoleGroup.value_counts()
display(Markdown(
    f"There are **{len(role_counts)} role families**; the smallest contains **{role_counts.min():,}** "
    "respondents. That overall count does not guarantee a large sample in each role-by-experience cell. "
    "The relationship notebook flags sparse cells before interpreting them."
))


## Distribution findings for the report
This text is calculated from the current prepared file. The relationship notebook
writes the combined EDA and Findings draft used for report assembly.


In [ ]:
row = overall.iloc[0]
text = (
    "# Jamal Part B distribution findings\n\n"
    f"The prepared sample contains {len(df):,} U.S. employed respondents. Median annual total "
    f"compensation is {dollars(row.median_usd)}, compared with mean {dollars(row.mean_usd)}. "
    f"The middle half runs from {dollars(row.q1_usd)} to {dollars(row.q3_usd)} "
    f"(IQR {dollars(row.iqr_usd)}). Compensation remains right-skewed after the preparation "
    "cutoffs; the raw and natural-log histograms describe the same retained respondents.\n\n"
    f"Median reported professional work experience is {experience.median():g} years. "
    f"Experience is missing for {df.ExperienceYears.isna().sum():,} respondents; these blanks "
    "may represent zero years or nonresponse. They remain visible in categorical counts "
    "and are excluded from numerical experience comparisons.\n\n"
    f"The nine prepared role families have at least {role_counts.min():,} respondents each. "
    "Work-arrangement plots show both the three prepared groups and the original five "
    "choices; the prepared Hybrid group includes Your choice. Missing work arrangement "
    f"is recorded for {info['missing_work_arrangement']} respondents and missing education "
    f"for {info['missing_education']}.\n\n"
    "These summaries describe the prepared sample, after excluding low and high pay in "
    "notebook 01. They do not describe the untrimmed distribution or establish full-time "
    "status. All source rows in the processed CSV are retained by notebooks 02 and 03.\n"
)
(output["report"] / "jamal_distributions.md").write_text(text, encoding="utf-8")
display(Markdown(text))
